In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from qcgpt.gates import VOCAB, PAD_ID, BOS_CIRC_ID, EOS_CIRC_ID
from qcgpt.models.policy import CircuitPolicy
from qcgpt.data.tasks import sample_task
from qcgpt.encoding import spec_to_tokens, tokens_to_circuit
from qcgpt.evaluation.metrics import (
    mapping_classical_accuracy,
    mapping_quantum_fidelity,
    gate_count,
)
from qcgpt.evaluation.visualize import format_mapping_spec, format_circuit


In [ ]:
def load_policy(ckpt_path, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    vocab_size = len(VOCAB)
    model = CircuitPolicy(
        vocab_size=vocab_size,
        d_model=256,
        n_layers=4,
        n_heads=4,
        max_spec_len=256,
        max_circ_len=128,
    ).to(device)

    if ckpt_path is not None and os.path.exists(ckpt_path):
        state = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(state["model_state_dict"])
        print(f"Loaded: {ckpt_path}")
    else:
        print(f"WARNING: checkpoint not found: {ckpt_path}")

    model.eval()
    return model


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

supervised_model = load_policy("checkpoints/supervised_epoch_010.pt", device)
rl_model         = load_policy("checkpoints/rl_finetuned.pt", device)


In [ ]:
@torch.no_grad()
def solve_task_with_model(model, spec_states, max_len=32, device=None):
    """
    Given a model and a spec_states [4,2,2], sample one circuit and compute metrics.
    """
    if device is None:
        device = next(model.parameters()).device

    spec_tokens = spec_to_tokens(spec_states)
    spec_tensor = torch.tensor(spec_tokens, dtype=torch.long, device=device).unsqueeze(0)

    sampled_tokens, log_probs = model.sample_circuit_tokens(
        spec_tokens=spec_tensor,
        bos_id=BOS_CIRC_ID,
        eos_id=EOS_CIRC_ID,
        max_len=max_len,
    )

    seq = sampled_tokens[0].tolist()
    seq = [t for t in seq if t != PAD_ID]
    circ = tokens_to_circuit(seq)

    acc = mapping_classical_accuracy(spec_states, circ)
    fid = mapping_quantum_fidelity(spec_states, circ)
    gc  = gate_count(circ)

    return circ, acc, fid, gc


In [ ]:
def evaluate_models_over_tasks(
    n_tasks=1000,
    max_gates_ref=6,
    max_len=32,
    device=None,
    eval_supervised=True,
    eval_rl=True,
):
    records = []
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for i in range(n_tasks):
        spec_states, ref_circ = sample_task(max_gates=max_gates_ref)

        # reference metrics
        acc_ref = mapping_classical_accuracy(spec_states, ref_circ)
        fid_ref = mapping_quantum_fidelity(spec_states, ref_circ)
        gc_ref  = gate_count(ref_circ)
        records.append({
            "task_id": i,
            "method": "reference",
            "acc_classical": acc_ref,
            "fid_quantum": fid_ref,
            "gate_count": gc_ref,
        })

        if eval_supervised and supervised_model is not None:
            circ_s, acc_s, fid_s, gc_s = solve_task_with_model(
                supervised_model, spec_states, max_len=max_len, device=device
            )
            records.append({
                "task_id": i,
                "method": "supervised",
                "acc_classical": acc_s,
                "fid_quantum": fid_s,
                "gate_count": gc_s,
            })

        if eval_rl and rl_model is not None:
            circ_r, acc_r, fid_r, gc_r = solve_task_with_model(
                rl_model, spec_states, max_len=max_len, device=device
            )
            records.append({
                "task_id": i,
                "method": "rl",
                "acc_classical": acc_r,
                "fid_quantum": fid_r,
                "gate_count": gc_r,
            })

        if (i+1) % 50 == 0:
            print(f"Evaluated {i+1}/{n_tasks} tasks")

    df = pd.DataFrame.from_records(records)
    return df


In [ ]:
os.makedirs("logs", exist_ok=True)
df.to_csv("logs/eval_tasks.csv", index=False)


In [ ]:
def plot_fidelity_hist(df):
    methods = df["method"].unique()
    plt.figure()
    for m in methods:
        subset = df[df["method"] == m]
        plt.hist(subset["fid_quantum"], bins=20, alpha=0.5, label=m)
    plt.xlabel("Quantum fidelity")
    plt.ylabel("Count")
    plt.title("Distribution of quantum fidelity across tasks")
    plt.legend()
    plt.show()

plot_fidelity_hist(df)


In [ ]:
def plot_gatecount_hist(df):
    methods = df["method"].unique()
    plt.figure()
    for m in methods:
        subset = df[df["method"] == m]
        plt.hist(subset["gate_count"], bins=range(0, 15), alpha=0.5, align="left", label=m)
    plt.xlabel("Gate count")
    plt.ylabel("Count")
    plt.title("Distribution of gate counts across tasks")
    plt.legend()
    plt.show()

plot_gatecount_hist(df)


In [ ]:
def plot_fidelity_vs_gatecount(df):
    methods = df["method"].unique()
    plt.figure()
    for m in methods:
        subset = df[df["method"] == m]
        plt.scatter(subset["gate_count"], subset["fid_quantum"], label=m, alpha=0.5)
    plt.xlabel("Gate count")
    plt.ylabel("Quantum fidelity")
    plt.title("Fidelity vs gate count")
    plt.legend()
    plt.show()

plot_fidelity_vs_gatecount(df)


In [ ]:
def find_interesting_examples(df, n_examples=5):
    # Example heuristic: tasks where RL gate_count < reference and RL fidelity ~ reference
    pivot = df.pivot(index="task_id", columns="method", values=["fid_quantum","gate_count"])
    rows = []
    for task_id in pivot.index:
        if "rl" not in pivot["fid_quantum"].columns:
            continue
        fid_ref = pivot["fid_quantum"].get("reference", {}).get(task_id, np.nan)
        fid_rl  = pivot["fid_quantum"].get("rl", {}).get(task_id, np.nan)
        gc_ref  = pivot["gate_count"].get("reference", {}).get(task_id, np.nan)
        gc_rl   = pivot["gate_count"].get("rl", {}).get(task_id, np.nan)

        if np.isnan([fid_ref, fid_rl, gc_ref, gc_rl]).any():
            continue

        # RL almost as good fidelity, but strictly fewer gates
        if fid_rl > 0.9 * fid_ref and gc_rl < gc_ref:
            rows.append((task_id, fid_ref, fid_rl, gc_ref, gc_rl))

    rows = sorted(rows, key=lambda x: (x[3]-x[4]), reverse=True)  # sort by gate reduction
    return rows[:n_examples]

interesting = find_interesting_examples(df, n_examples=5)
interesting


In [ ]:
for (task_id, fid_ref, fid_rl, gc_ref, gc_rl) in interesting:
    spec_states, ref_circ = sample_task(max_gates=6)  # careful! this re-samples; for real reproducibility you'd store seeds/specs

    print("="*80)
    print(f"Task {task_id}")
    print("Mapping spec:")
    print(format_mapping_spec(spec_states))
    print("-"*40)
    print("Reference circuit:")
    print(format_circuit(ref_circ))
    print(f"  Gate count: {gc_ref}")
    circ_rl, acc_rl, fid_rl, gc_rl = solve_task_with_model(rl_model, spec_states, max_len=32, device=device)
    print("-"*40)
    print("RL circuit:")
    print(format_circuit(circ_rl))
    print(f"  Gate count: {gc_rl}  |  Fidelity: {fid_rl:.3f}")
